In [1]:
import sys
import os

from pyflamegpu import *
import pyflamegpu.codegen
import sys
# 切换到你的项目根目录


In [2]:
from deap import algorithms
from deap import base
from deap import creator
from deap import tools

In [3]:

project_root = r"D:\progamming\python_script\Social-physical-system-shelter"
os.chdir(project_root)

# 添加 data/output 目录到 Python 路径
sys.path.append('data/output')

In [4]:
from pyflamegpu import *
import pyflamegpu.codegen
import sys
import math
import random

model = pyflamegpu.ModelDescription("F_MAP_tutorial")


# Define an host function called directed_graph_hostfn
class directed_graph_hostfn(pyflamegpu.HostFunction):
  def run(self,FLAMEGPU):
    # Fetch a handle to the directed graph
    fgraph = FLAMEGPU.environment.getDirectedGraph("fgraph")
    # Import a different graph
    fgraph.importGraph("data/env_data/expanded_visibility_graph_renumbered.json");








In [5]:
# Define an host function called write_env_hostfn
class write_env_hostfn(pyflamegpu.HostFunction):
  
  def __init__(self):
    super().__init__()  
  
  def run(self,FLAMEGPU):

      # Retrieve the environment macro property bar of type int array[5][5]

      # Update some of the values
      # foo = 12.0; is not allowed
      FLAMEGPU.environment.importMacroProperty("map", "data/env_data/attraction_matrix_familiar_points.json");

## message

In [6]:

# Define a message of type MessageSpatial2D named location
# MessageSpatial2D: Each agent outputs a message at a specific location in 2D space
# agents only read messages located close to a particular search origin（搜素的中心点）.
# 可以获取一定距离内的消息
message = model.newMessageSpatial3D("location")
# Configure the message list

message.setMin(0, 0,0)
message.setMax(500, 500,100)
message.setRadius(200)
# Add extra variables to the message
# X Y (Z) are implicit for spatial messages
message.newVariableID("id")



In [7]:
stairwell_message = model.newMessageSpatial3D("stairwell_location")
# Configure the message list

stairwell_message.setMin(0, 0,0)
stairwell_message.setMax(500, 500,100)
stairwell_message.setRadius(200)
# Add extra variables to the message
# X Y (Z) are implicit for spatial messages
stairwell_message.newVariableID("id")
stairwell_message.newVariableInt("building_id")
stairwell_message.newVariableFloat("outstop_x")
stairwell_message.newVariableFloat("outstop_y")
stairwell_message.newVariableInt("graph_id")

In [8]:
shelter_message = model.newMessageSpatial3D("shelter_location")
shelter_message.setMin(0, 0,0)
shelter_message.setMax(1000, 1000,200)
shelter_message.setRadius(1000)
shelter_message.newVariableID("id")
shelter_message.newVariableInt("shelter_id")
shelter_message.newVariableInt("graph_id")
# 容量机制：1=可用；0=满员/不可用
shelter_message.newVariableInt("available")

# 到达统计：student/tutor 在“被 shelter 接纳”时发出到达消息（用于 shelter 计数）
arrival_message = model.newMessageSpatial3D("arrival")
arrival_message.setMin(0, 0, 0)
arrival_message.setMax(1000, 1000, 200)
arrival_message.setRadius(2)  # 只统计几乎同一点的到达（避免误计）
arrival_message.newVariableID("id")
arrival_message.newVariableInt("shelter_id")

# tutor 引导信息：3D 空间消息，小范围传播（只有附近 student 才能收到）
tutor_guidance_message = model.newMessageSpatial3D("tutor_guidance")
tutor_guidance_message.setMin(0, 0, 0)
tutor_guidance_message.setMax(1000, 1000, 200)
tutor_guidance_message.setRadius(5)  # 尽量小的范围（可按需要调 10/20/30）
tutor_guidance_message.newVariableID("id")
tutor_guidance_message.newVariableInt("shelter_id")
tutor_guidance_message.newVariableInt("end_id")
tutor_guidance_message.newVariableFloat("shelter_x")
tutor_guidance_message.newVariableFloat("shelter_y")


In [9]:
# Assign the agent some variables (ID is implicit to agents, so we don't define it ourselves)
student_agent = model.newAgent("student_agent")
student_agent.newVariableFloat("x")
student_agent.newVariableFloat("y")
student_agent.newVariableInt("building_id")
student_agent.newVariableInt("point_id")
student_agent.newVariableFloat("z")
student_agent.newVariableFloat("drift", 0)
student_agent.newVariableInt("target_stairwell_id", -1)
student_agent.newVariableFloat("target_stairwell_x")
student_agent.newVariableFloat("target_stairwell_y")
student_agent.newVariableInt("target_shelter_id", -1)
student_agent.newVariableInt("is_set_shelter", -1)
student_agent.newVariableInt("evacuate_status", 1)
student_agent.newVariableInt("zigzag_dir", 1)
student_agent.newVariableFloat("target_shelter_x")
student_agent.newVariableFloat("target_shelter_y")
# stairwell-connected out stop point:
student_agent.newVariableFloat("outstop_x")
student_agent.newVariableFloat("outstop_y")

student_agent.newVariableInt("start_id")
student_agent.newVariableInt("end_id")
# for road planning





student_agent.newVariableFloat("bar_0_0")
student_agent.newVariableInt("start_vertex_id",131)
student_agent.newVariableInt("end_vertex_id",175)
student_agent.newVariableInt("path_length")
student_agent.newVariableArrayInt("shortest_path", 20)  # 存储最短路径的顶点ID数组，16个顶点的图最长路径不超过20





student_agent.newVariableArrayFloat("shortest_value",176, [-1.0] * 176)

# Dijkstra算法需要的变量
student_agent.newVariableArrayFloat("distances", 176, [999999.0] * 176)  # 距离数组，初始化为无穷大
student_agent.newVariableArrayInt("visited", 176, [0] * 176)  # 访问标记数组
student_agent.newVariableArrayInt("previous", 176, [-1] * 176)  # 前驱节点数组
student_agent.newVariableInt("vertex_count")  # 顶点数量
student_agent.newVariableInt("edge_count")  # 边数量
student_agent.newVariableInt("is_set_shortest_path", 0)

#for move
student_agent.newVariableInt("path_point", 1)

# 容量机制：到达 shelter 后是否被接纳 / 是否已上报到达（避免重复计数）
student_agent.newVariableInt("accepted", 0)
student_agent.newVariableInt("arrival_sent", 0)


In [10]:
# Assign the agent some variables (ID is implicit to agents, so we don't define it ourselves)
tutor_agent = model.newAgent("tutor_agent")
tutor_agent.newVariableFloat("x")
tutor_agent.newVariableFloat("y")
tutor_agent.newVariableInt("building_id")
tutor_agent.newVariableInt("point_id")
tutor_agent.newVariableFloat("z")
tutor_agent.newVariableFloat("drift", 0)
tutor_agent.newVariableInt("target_stairwell_id", -1)
tutor_agent.newVariableFloat("target_stairwell_x")
tutor_agent.newVariableFloat("target_stairwell_y")
tutor_agent.newVariableInt("target_shelter_id", -1)
tutor_agent.newVariableInt("is_set_shelter", -1)
tutor_agent.newVariableInt("evacuate_status", 1)
tutor_agent.newVariableInt("zigzag_dir", 1)
tutor_agent.newVariableFloat("target_shelter_x")
tutor_agent.newVariableFloat("target_shelter_y")
# stairwell-connected out stop point:
tutor_agent.newVariableFloat("outstop_x")
tutor_agent.newVariableFloat("outstop_y")

tutor_agent.newVariableInt("start_id")
tutor_agent.newVariableInt("end_id")
# for road planning

tutor_agent.newVariableFloat("bar_0_0")
tutor_agent.newVariableInt("start_vertex_id",131)
tutor_agent.newVariableInt("end_vertex_id",175)
tutor_agent.newVariableInt("path_length")
tutor_agent.newVariableArrayInt("shortest_path", 20)  # 存储最短路径的顶点ID数组，16个顶点的图最长路径不超过20

tutor_agent.newVariableArrayFloat("shortest_value",176, [-1.0] * 176)

# Dijkstra算法需要的变量
tutor_agent.newVariableArrayFloat("distances", 176, [999999.0] * 176)  # 距离数组，初始化为无穷大
tutor_agent.newVariableArrayInt("visited", 176, [0] * 176)  # 访问标记数组
tutor_agent.newVariableArrayInt("previous", 176, [-1] * 176)  # 前驱节点数组
tutor_agent.newVariableInt("vertex_count")  # 顶点数量
tutor_agent.newVariableInt("edge_count")  # 边数量
tutor_agent.newVariableInt("is_set_shortest_path", 0)

#for move
tutor_agent.newVariableInt("path_point", 1)

# 容量机制：到达 shelter 后是否被接纳 / 是否已上报到达（避免重复计数）
tutor_agent.newVariableInt("accepted", 0)
tutor_agent.newVariableInt("arrival_sent", 0)


In [11]:
stairwell_agent = model.newAgent("stairwell_agent")
stairwell_agent.newVariableFloat("x")
stairwell_agent.newVariableFloat("y")
stairwell_agent.newVariableInt("stairwell_id")
stairwell_agent.newVariableInt("building_id")
stairwell_agent.newVariableFloat("outstop_x")
stairwell_agent.newVariableFloat("outstop_y")
stairwell_agent.newVariableInt("graph_id")
stairwell_agent.newVariableFloat("z")


In [12]:
shelter_agent = model.newAgent("shelter_agent")
shelter_agent.newVariableFloat("x")
shelter_agent.newVariableFloat("y")
shelter_agent.newVariableInt("shelter_id")
shelter_agent.newVariableFloat("z")
shelter_agent.newVariableInt("graph_id")

# 容量机制：当前已接纳人数 & 是否可用（1=可用；0=满员）
shelter_agent.newVariableInt("occupancy", 0)
shelter_agent.newVariableInt("available", 1)


In [13]:


# Fetch the model's environment
env = model.Environment()
# Declare a new directed graph named 'fgraph'
fgraph = env.newDirectedGraph("fgraph")
# Attach an float[2] property 'bar' to vertices
fgraph.newVertexPropertyArrayFloat("bar", 2)
# Attach an int property 'foo' to edges
fgraph.newEdgePropertyFloat("foo")
env.newMacroPropertyFloat("map", 700, 700)

# 首先我要import一个graph



In [14]:

@pyflamegpu.agent_function
def output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    return pyflamegpu.ALIVE


In [15]:
@pyflamegpu.agent_function
def stairwell_output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setVariableInt("building_id", pyflamegpu.getVariableInt("building_id"))
    message_out.setVariableFloat("outstop_x", pyflamegpu.getVariableFloat("outstop_x"))
    message_out.setVariableFloat("outstop_y", pyflamegpu.getVariableFloat("outstop_y"))
    message_out.setVariableInt("graph_id", pyflamegpu.getVariableInt("graph_id"))
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    return pyflamegpu.ALIVE

In [16]:
@pyflamegpu.agent_function
def shelter_output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setVariableInt("shelter_id", pyflamegpu.getVariableInt("shelter_id"))
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    message_out.setVariableInt("graph_id", pyflamegpu.getVariableInt("graph_id"))
    message_out.setVariableInt("available", pyflamegpu.getVariableInt("available"))
    return pyflamegpu.ALIVE

In [17]:
@pyflamegpu.agent_function
def set_target_stairwell(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    # Get this agent's x, y, z variables
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")
    target_stairwell_id = pyflamegpu.getVariableInt("target_stairwell_id")

    min_dist = 100000
    if target_stairwell_id == -1:
        for message in message_in(x,y,z):
            # Process the message's variables e.g.
            if message.getVariableInt("building_id") == pyflamegpu.getVariableInt("building_id"):
                #找到距离最近的楼梯
                # 找到距离最近的楼梯

                stairwell_x = message.getVariableFloat("x")
                stairwell_y = message.getVariableFloat("y")

                # 计算欧氏距离
                dx = stairwell_x - x
                dy = stairwell_y - y

                dist = math.sqrtf(dx*dx + dy*dy)
                if dist < min_dist:
                    min_dist = dist
                    nearest_stairwell_id = message.getVariableInt("id")
                    # 记录最近楼梯的坐标
                    pyflamegpu.setVariableInt("target_stairwell_id", nearest_stairwell_id)
                    pyflamegpu.setVariableFloat("target_stairwell_x", stairwell_x)
                    pyflamegpu.setVariableFloat("target_stairwell_y", stairwell_y)
                    pyflamegpu.setVariableInt("start_id", message.getVariableInt("graph_id"))
                    pyflamegpu.setVariableFloat("outstop_x", message.getVariableFloat("outstop_x"))
                    pyflamegpu.setVariableFloat("outstop_y", message.getVariableFloat("outstop_y")) 

            #设置目标楼梯
    return pyflamegpu.ALIVE

In [18]:
@pyflamegpu.agent_function
def set_target_shelter_first(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    """按权重概率选择 shelter。

    权重 w = familiarity(m,n) / dist
    P(选择 i) = w_i / sum(w)

    注意：设备端不适合存整张概率表，这里用“累计权重抽样”。
    """
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")

    if pyflamegpu.getVariableInt("is_set_shelter") != -1:
        return pyflamegpu.ALIVE

    # 只取一次 map（避免循环内重复读取）
    fmap = pyflamegpu.environment.getMacroPropertyFloat("map", 700, 700)

    total_w = 0.0
    # --- pass 1: 求权重总和 ---
    for message in message_in(x, y, z):
        # 容量机制：跳过满员 shelter
        if message.getVariableInt("available") == 0:
            continue

        shelter_x = message.getVariableFloat("x")
        shelter_y = message.getVariableFloat("y")

        dx = shelter_x - x
        dy = shelter_y - y
        dist = math.sqrtf(dx * dx + dy * dy)
        if dist <= 1e-6:
            continue

        m = int(shelter_x)
        n = int(shelter_y)
        # 防止越界（shelter 坐标/地图边界不一致时）
        if m < 0:
            m = 0
        elif m > 699:
            m = 699
        if n < 0:
            n = 0
        elif n > 699:
            n = 699

        familiarity = math.sqrtf(fmap[m][n]) + 100.0
        w = familiarity / dist
        if w > 0.0:
            total_w += w

    # 没有候选 shelter（或全部权重为 0）就保持不变
    if total_w <= 0.0:
        return pyflamegpu.ALIVE

    # --- pass 2: 按累计权重抽样 ---
    r = pyflamegpu.random.uniformFloat() * total_w
    cum_w = 0.0

    for message in message_in(x, y, z):
        if message.getVariableInt("available") == 0:
            continue

        shelter_x = message.getVariableFloat("x")
        shelter_y = message.getVariableFloat("y")

        dx = shelter_x - x
        dy = shelter_y - y
        dist = math.sqrtf(dx * dx + dy * dy)
        if dist <= 1e-6:
            continue

        m = int(shelter_x)
        n = int(shelter_y)
        if m < 0:
            m = 0
        elif m > 699:
            m = 699
        if n < 0:
            n = 0
        elif n > 699:
            n = 699

        familiarity = math.sqrtf(fmap[m][n]) + 100.0
        w = familiarity / dist
        if w <= 0.0:
            continue

        cum_w += w
        if cum_w >= r:
            pyflamegpu.setVariableInt("target_shelter_id", message.getVariableInt("shelter_id"))
            pyflamegpu.setVariableFloat("target_shelter_x", shelter_x)
            pyflamegpu.setVariableFloat("target_shelter_y", shelter_y)
            pyflamegpu.setVariableInt("is_set_shelter", 1)
            pyflamegpu.setVariableInt("end_id", message.getVariableInt("graph_id"))
            break

    return pyflamegpu.ALIVE


# =========================
# tutor_agent 的 set 逻辑
# =========================

@pyflamegpu.agent_function
def tutor_set_target_stairwell(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    """tutor 一开始先选同楼最近楼梯，并写入 start_id/outstop"""
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")

    target_stairwell_id = pyflamegpu.getVariableInt("target_stairwell_id")
    if target_stairwell_id != -1:
        return pyflamegpu.ALIVE

    min_dist = 999999.0
    for message in message_in(x, y, z):
        if message.getVariableInt("building_id") != pyflamegpu.getVariableInt("building_id"):
            continue

        stairwell_x = message.getVariableFloat("x")
        stairwell_y = message.getVariableFloat("y")
        dx = stairwell_x - x
        dy = stairwell_y - y
        dist = math.sqrtf(dx * dx + dy * dy)

        if dist < min_dist:
            min_dist = dist
            nearest_stairwell_id = message.getVariableInt("id")
            pyflamegpu.setVariableInt("target_stairwell_id", nearest_stairwell_id)
            pyflamegpu.setVariableFloat("target_stairwell_x", stairwell_x)
            pyflamegpu.setVariableFloat("target_stairwell_y", stairwell_y)

            # stairwell_message.graph_id 是 outstop 点匹配到的路网节点 id
            pyflamegpu.setVariableInt("start_id", message.getVariableInt("graph_id"))
            pyflamegpu.setVariableFloat("outstop_x", message.getVariableFloat("outstop_x"))
            pyflamegpu.setVariableFloat("outstop_y", message.getVariableFloat("outstop_y"))

    return pyflamegpu.ALIVE


@pyflamegpu.agent_function
def tutor_set_target_shelter_by_network(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    """tutor 使用路网最短路距离选择最近 shelter（与 student 的熟悉度/欧氏不同）"""
    if pyflamegpu.getVariableInt("is_set_shelter") != -1:
        return pyflamegpu.ALIVE

    start_vertex_id = pyflamegpu.getVariableInt("start_id")
    if start_vertex_id < 0:
        return pyflamegpu.ALIVE

    fgraph = pyflamegpu.environment.getDirectedGraph("fgraph")

    INF = 999999.0
    vertex_count = 176

    best_dist = INF
    best_shelter_id = -1
    best_end_vertex_id = -1
    best_x = 0.0
    best_y = 0.0

    outstop_x = pyflamegpu.getVariableFloat("outstop_x")
    outstop_y = pyflamegpu.getVariableFloat("outstop_y")

    # 以 outstop 为中心读取 shelter messages（radius 足够大时可以拿到所有 shelter）
    for message in message_in(outstop_x, outstop_y, 0.0):
        # 容量机制：跳过满员 shelter
        if message.getVariableInt("available") == 0:
            continue

        end_vertex_id = message.getVariableInt("graph_id")

        # --- Dijkstra: 只算距离，不重建路径（用于选择最近 shelter） ---
        start_index = fgraph.getVertexIndex(start_vertex_id)
        end_index = fgraph.getVertexIndex(end_vertex_id)

        for i in range(vertex_count):
            pyflamegpu.setVariableFloatArray176("distances", i, INF)
            pyflamegpu.setVariableIntArray176("visited", i, 0)

        pyflamegpu.setVariableFloatArray176("distances", start_index, 0.0)

        for _ in range(vertex_count):
            min_distance = INF
            current_index = -1

            for i in range(vertex_count):
                visited_i = pyflamegpu.getVariableIntArray176("visited", i)
                distance_i = pyflamegpu.getVariableFloatArray176("distances", i)
                if visited_i == 0 and distance_i < min_distance:
                    min_distance = distance_i
                    current_index = i

            if current_index == -1 or min_distance == INF:
                break

            if current_index == end_index:
                break

            pyflamegpu.setVariableIntArray176("visited", current_index, 1)

            for edge in fgraph.outEdges(current_index):
                dest_index = edge.getEdgeDestination()
                visited_dest = pyflamegpu.getVariableIntArray176("visited", dest_index)
                if visited_dest == 0:
                    edge_weight = edge.getPropertyFloat("foo")
                    new_distance = min_distance + edge_weight
                    dest_distance = pyflamegpu.getVariableFloatArray176("distances", dest_index)
                    if new_distance < dest_distance:
                        pyflamegpu.setVariableFloatArray176("distances", dest_index, new_distance)

        dist_to_end = pyflamegpu.getVariableFloatArray176("distances", end_index)

        if dist_to_end < best_dist:
            best_dist = dist_to_end
            best_shelter_id = message.getVariableInt("shelter_id")
            best_end_vertex_id = end_vertex_id
            best_x = message.getVariableFloat("x")
            best_y = message.getVariableFloat("y")

    if best_shelter_id != -1 and best_dist < INF:
        pyflamegpu.setVariableInt("target_shelter_id", best_shelter_id)
        pyflamegpu.setVariableFloat("target_shelter_x", best_x)
        pyflamegpu.setVariableFloat("target_shelter_y", best_y)
        pyflamegpu.setVariableInt("end_id", best_end_vertex_id)
        pyflamegpu.setVariableInt("is_set_shelter", 1)

    return pyflamegpu.ALIVE


# =========================
# tutor 广播 + student 接收引导
# =========================

@pyflamegpu.agent_function
def tutor_broadcast_shelter(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    """tutor 在到达 outstop 前（evacuate_status < 4）持续广播自己选择的 shelter"""
    # 约束：到达 outstop（evacuate_status >= 4）后不再广播
    if pyflamegpu.getVariableInt("evacuate_status") >= 4:
        return pyflamegpu.ALIVE

    # 还没选到 shelter 就不发
    if pyflamegpu.getVariableInt("is_set_shelter") != 1:
        return pyflamegpu.ALIVE

    sx = pyflamegpu.getVariableFloat("x")
    sy = pyflamegpu.getVariableFloat("y")
    sz = pyflamegpu.getVariableFloat("z")

    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setVariableInt("shelter_id", pyflamegpu.getVariableInt("target_shelter_id"))
    message_out.setVariableInt("end_id", pyflamegpu.getVariableInt("end_id"))
    message_out.setVariableFloat("shelter_x", pyflamegpu.getVariableFloat("target_shelter_x"))
    message_out.setVariableFloat("shelter_y", pyflamegpu.getVariableFloat("target_shelter_y"))
    message_out.setLocation(sx, sy, sz)
    return pyflamegpu.ALIVE


@pyflamegpu.agent_function
def student_follow_tutor(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    """student 在到达 outstop 前可根据附近 tutor 广播修改自己的疏散 shelter

    约束：当 student 到达 outstop 后（evacuate_status >= 4）不再改变选择。
    """
    if pyflamegpu.getVariableInt("evacuate_status") >= 4:
        return pyflamegpu.ALIVE

    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")

    best_d = 999999.0
    best_shelter_id = -1
    best_end_id = -1
    best_sx = 0.0
    best_sy = 0.0

    # 只接收小范围内的 tutor_guidance（message radius 已经很小）
    for msg in message_in(x, y, z):
        tx = msg.getVariableFloat("x")
        ty = msg.getVariableFloat("y")
        tz = msg.getVariableFloat("z")
        dx = tx - x
        dy = ty - y
        dz = tz - z
        d = math.sqrtf(dx * dx + dy * dy + dz * dz)
        if d < best_d:
            best_d = d
            best_shelter_id = msg.getVariableInt("shelter_id")
            best_end_id = msg.getVariableInt("end_id")
            best_sx = msg.getVariableFloat("shelter_x")
            best_sy = msg.getVariableFloat("shelter_y")

    if best_shelter_id == -1:
        return pyflamegpu.ALIVE

    # 如果 tutor 的建议与当前不同，则更新，并重置最短路（使 ShortestPathFn 重新跑）
    if best_shelter_id != pyflamegpu.getVariableInt("target_shelter_id"):
        pyflamegpu.setVariableInt("target_shelter_id", best_shelter_id)
        pyflamegpu.setVariableFloat("target_shelter_x", best_sx)
        pyflamegpu.setVariableFloat("target_shelter_y", best_sy)
        pyflamegpu.setVariableInt("end_id", best_end_id)
        pyflamegpu.setVariableInt("is_set_shelter", 1)

        # 关键：允许改选 -> 重置 shortest path
        pyflamegpu.setVariableInt("is_set_shortest_path", 0)
        pyflamegpu.setVariableInt("path_point", 1)
        pyflamegpu.setVariableInt("path_length", 0)

    return pyflamegpu.ALIVE


# =========================
# shelter 容量：到达计数 + 满员广播 + 到达后改选重规划
# =========================




@pyflamegpu.agent_function
def shelter_update_occupancy(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    """shelter 统计本步新到达人数，并更新 available（满员后 available=0）"""
    sid = pyflamegpu.getVariableInt("shelter_id")
    sx = pyflamegpu.getVariableFloat("x")
    sy = pyflamegpu.getVariableFloat("y")
    sz = pyflamegpu.getVariableFloat("z")

    added = 0
    for msg in message_in(sx, sy, sz):
        if msg.getVariableInt("shelter_id") == sid:
            added += 1

    occ = pyflamegpu.getVariableInt("occupancy") + added
    pyflamegpu.setVariableInt("occupancy", occ)
    
    SHELTER_CAPACITY = 1200
    if occ >= SHELTER_CAPACITY:
        pyflamegpu.setVariableInt("available", 0)
    else:
        pyflamegpu.setVariableInt("available", 1)

    return pyflamegpu.ALIVE


@pyflamegpu.agent_function
def check_shelter_capacity_and_replan(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    """student/tutor 到达 shelter(=evacuate_status==5) 时：
    - 若目标 shelter 可用：标记 accepted=1（下一步会发 arrival 消息给 shelter 计数）
    - 若满员：改选“路网距离最近的可用 shelter”，并重置最短路，让下一步继续走
    """
    # 只在到达 shelter 后处理
    if pyflamegpu.getVariableInt("evacuate_status") != 5:
        return pyflamegpu.ALIVE

    # 已接纳/已处理过就不重复做
    if pyflamegpu.getVariableInt("accepted") == 1:
        return pyflamegpu.ALIVE

    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")

    my_shelter_id = pyflamegpu.getVariableInt("target_shelter_id")

    # 找到“当前目标 shelter”的可用状态
    current_available = -1
    for smsg in message_in(x, y, z):
        if smsg.getVariableInt("shelter_id") == my_shelter_id:
            current_available = smsg.getVariableInt("available")
            break

    # 可用：接纳
    if current_available == 1:
        pyflamegpu.setVariableInt("accepted", 1)
        return pyflamegpu.ALIVE

    # 满员：改选
    fgraph = pyflamegpu.environment.getDirectedGraph("fgraph")

    INF = 999999.0
    vertex_count = 176

    # 当前所处 shelter 对应的路网点（上一次 end_id）
    start_vertex_id = pyflamegpu.getVariableInt("end_id")
    if start_vertex_id < 0:
        return pyflamegpu.ALIVE

    best_dist = INF
    best_shelter_id = -1
    best_end_vertex_id = -1
    best_x = 0.0
    best_y = 0.0

    start_index = fgraph.getVertexIndex(start_vertex_id)

    for cand in message_in(x, y, z):
        if cand.getVariableInt("available") == 0:
            continue

        cand_shelter_id = cand.getVariableInt("shelter_id")
        cand_end_vertex_id = cand.getVariableInt("graph_id")

        # 避免选回同一个满员 shelter
        if cand_shelter_id == my_shelter_id:
            continue

        end_index = fgraph.getVertexIndex(cand_end_vertex_id)

        # init arrays
        for i in range(vertex_count):
            pyflamegpu.setVariableFloatArray176("distances", i, INF)
            pyflamegpu.setVariableIntArray176("visited", i, 0)

        pyflamegpu.setVariableFloatArray176("distances", start_index, 0.0)

        # Dijkstra（只算距离）
        for _ in range(vertex_count):
            min_distance = INF
            current_index = -1

            for i in range(vertex_count):
                visited_i = pyflamegpu.getVariableIntArray176("visited", i)
                distance_i = pyflamegpu.getVariableFloatArray176("distances", i)
                if visited_i == 0 and distance_i < min_distance:
                    min_distance = distance_i
                    current_index = i

            if current_index == -1 or min_distance == INF:
                break

            if current_index == end_index:
                break

            pyflamegpu.setVariableIntArray176("visited", current_index, 1)

            for edge in fgraph.outEdges(current_index):
                dest_index = edge.getEdgeDestination()
                if pyflamegpu.getVariableIntArray176("visited", dest_index) == 0:
                    edge_weight = edge.getPropertyFloat("foo")
                    new_distance = min_distance + edge_weight
                    dest_distance = pyflamegpu.getVariableFloatArray176("distances", dest_index)
                    if new_distance < dest_distance:
                        pyflamegpu.setVariableFloatArray176("distances", dest_index, new_distance)

        cand_dist = pyflamegpu.getVariableFloatArray176("distances", end_index)
        if cand_dist < best_dist:
            best_dist = cand_dist
            best_shelter_id = cand_shelter_id
            best_end_vertex_id = cand_end_vertex_id
            best_x = cand.getVariableFloat("x")
            best_y = cand.getVariableFloat("y")

    if best_shelter_id != -1 and best_dist < INF:
        # 从当前点重新出发：start_id = 当前位置的路网点
        pyflamegpu.setVariableInt("start_id", start_vertex_id)

        pyflamegpu.setVariableInt("target_shelter_id", best_shelter_id)
        pyflamegpu.setVariableFloat("target_shelter_x", best_x)
        pyflamegpu.setVariableFloat("target_shelter_y", best_y)
        pyflamegpu.setVariableInt("end_id", best_end_vertex_id)
        pyflamegpu.setVariableInt("is_set_shelter", 1)

        # 重置最短路，让 ShortestPathFn 下一步重新跑
        pyflamegpu.setVariableInt("is_set_shortest_path", 0)
        pyflamegpu.setVariableInt("path_point", 1)
        pyflamegpu.setVariableInt("path_length", 0)

        # 回到路网移动阶段
        pyflamegpu.setVariableInt("evacuate_status", 4)

        # 重置到达标记
        pyflamegpu.setVariableInt("accepted", 0)
        pyflamegpu.setVariableInt("arrival_sent", 0)

    return pyflamegpu.ALIVE


@pyflamegpu.agent_function
def output_arrival_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    """被接纳后，上报到达（用于 shelter 计数）"""
    if pyflamegpu.getVariableInt("accepted") != 1:
        return pyflamegpu.ALIVE
    if pyflamegpu.getVariableInt("arrival_sent") == 1:
        return pyflamegpu.ALIVE

    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")

    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setVariableInt("shelter_id", pyflamegpu.getVariableInt("target_shelter_id"))
    message_out.setLocation(x, y, z)

    pyflamegpu.setVariableInt("arrival_sent", 1)
    # 标记完成态（不再移动）
    pyflamegpu.setVariableInt("evacuate_status", 6)

    return pyflamegpu.ALIVE




In [19]:

### 好消息，根据我的test_set_get_in_onefunc.py，我可以实现动态的数据存储啦！！！
@pyflamegpu.agent_function
def ShortestPathFn(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageNone):
    """
    使用Dijkstra算法实现最短路径规划
    """
    fgraph = pyflamegpu.environment.getDirectedGraph("fgraph")

    # 定义常量
    INF = 999999.0
    vertex_count = 176  # 假设图中有16个顶点

    # 获取起点和终点ID（可以从agent变量中获取，或者设置为固定值）
    start_vertex_id = pyflamegpu.getVariableInt("end_id")
    end_vertex_id = pyflamegpu.getVariableInt("start_id")

    ###
    # 起点是哪个，我们就把它的list对应的i，设置为0。
    ###
    pyflamegpu.setVariableFloatArray176("shortest_value", start_vertex_id, 0)

    #然后遍历所有顶点（1-16），除了我们的起点id外，如果它和起点id有连线，我们就把它的list对应的i，设置为起点到它的距离。

    #遍历起点的id，线的值就设置为shortest_value的起点对应的i的值+边的值。
    # 先初始化所有顶点的shortest_value为INF
    for i in range(176):
        if i != start_vertex_id:
            pyflamegpu.setVariableFloatArray176("shortest_value", i, INF)

    # 遍历起点连接的边，设置对应的shortest_value
    for edge in fgraph.outEdges(start_vertex_id):
        # 获取边的目的顶点索引
        dest_vertex_index = edge.getEdgeDestination()
        # 获取边的foo属性
        foo = edge.getPropertyInt("foo")

        # 将foo值存储到对应顶点ID的位置
        pyflamegpu.setVariableFloatArray176("shortest_value", dest_vertex_index, foo)
            


    # 获取起点和终点的索引
    start_index = fgraph.getVertexIndex(start_vertex_id)
    end_index = fgraph.getVertexIndex(end_vertex_id)

    # 初始化起点距离为0
    pyflamegpu.setVariableFloatArray176("distances", start_index, 0.0)
    # 初始化起点前驱为-1
    pyflamegpu.setVariableIntArray176("previous", start_index, -1)
    # 初始化其他顶点的距离为无穷大，前驱为-1
    for i in range(vertex_count):
        if i != start_index:
            pyflamegpu.setVariableFloatArray176("distances", i, INF)
            pyflamegpu.setVariableIntArray176("previous", i, -1)
        pyflamegpu.setVariableIntArray176("visited", i, 0)

    # Dijkstra算法主循环
    for _ in range(vertex_count):
        # 找到当前未访问顶点中距离最小的顶点
        min_distance = INF      
        current_index = -1

        for i in range(vertex_count):
            visited_i = pyflamegpu.getVariableIntArray176("visited", i)
            distance_i = pyflamegpu.getVariableFloatArray176("distances", i) 
            if visited_i == 0 and distance_i < min_distance:
                min_distance = distance_i
                current_index = i

        if current_index == -1 or min_distance == INF:
            break

        # 标记当前顶点为已访问
        pyflamegpu.setVariableIntArray176("visited", current_index, 1)   

        # 如果到达终点，可以提前结束
        if current_index == end_index:
            break


        # 遍历当前顶点的所有出边
        for edge in fgraph.outEdges(current_index):

            dest_index = edge.getEdgeDestination()

            visited_dest = pyflamegpu.getVariableIntArray176("visited", dest_index)
            if visited_dest == 0:
                # 获取边的权重
                edge_weight = edge.getPropertyFloat("foo")
                current_distance = pyflamegpu.getVariableFloatArray176("distances", current_index)
                new_distance = current_distance + edge_weight

                dest_distance = pyflamegpu.getVariableFloatArray176("distances", dest_index)
                if new_distance < dest_distance:
                    pyflamegpu.setVariableFloatArray176("distances", dest_index, new_distance)
                    pyflamegpu.setVariableIntArray176("previous", dest_index, current_index)

    # 重建最短路径 - 使用pyflamegpu方法完全避免循环和索引操作

    # 初始化路径数组为-1
    for i in range(20):
        pyflamegpu.setVariableIntArray20("shortest_path", i, -1)

    # 从终点开始重建路径
    current = end_index
    path_index = 0
    path_length = 0

    # 逆向追溯直到起点或前驱为-1
    while current != -1 and path_index < 20:
        vertex_id = fgraph.getVertexID(current)
        pyflamegpu.setVariableIntArray20("shortest_path", path_index, vertex_id)
        path_index += 1
        path_length += 1
        
        # 如果到达起点，停止追溯
        if current == start_index:  # 假设start_index是起点的索引
            break
        
        # 获取前驱节点
        current = pyflamegpu.getVariableIntArray176("previous", current)

    # 设置路径长度
    pyflamegpu.setVariableInt("path_length", path_length)

    # 验证路径是否确实从终点连接到起点
    first_vertex = pyflamegpu.getVariableIntArray20("shortest_path", 0)
    last_vertex = pyflamegpu.getVariableIntArray20("shortest_path", path_length - 1)

    if first_vertex != end_vertex_id or last_vertex != start_vertex_id:
        # 路径不完整，可能需要特殊处理
        pyflamegpu.setVariableInt("path_length", 0)  # 或者标记为无效路径

    pyflamegpu.setVariableInt("is_set_shortest_path", 1)
    return pyflamegpu.ALIVE 



In [20]:
@pyflamegpu.agent_function_condition
def is_set_shortest_path_is_1() -> bool:
    return pyflamegpu.getVariableInt("is_set_shortest_path") == 0

In [21]:
import math

In [22]:
@pyflamegpu.agent_function
def move_to_stairwell(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    target_stairwell_x = pyflamegpu.getVariableFloat("target_stairwell_x")
    target_stairwell_y = pyflamegpu.getVariableFloat("target_stairwell_y")
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    
    delta_x = target_stairwell_x - x
    delta_y = target_stairwell_y - y

    distance = math.sqrtf(delta_x * delta_x + delta_y * delta_y)

    next_x=0.0
    next_y=0.0
    if distance > 5.0:
        step_x = delta_x / distance 
        step_y = delta_y / distance
        next_x = x + 5*step_x
        next_y = y + 5*step_y
    else:
        next_x =  target_stairwell_x
        next_y =  target_stairwell_y
        pyflamegpu.setVariableInt("evacuate_status", 2)

    pyflamegpu.setVariableFloat("x", next_x)
    pyflamegpu.setVariableFloat("y", next_y)

    
    message_out.setLocation(next_x, next_y, pyflamegpu.getVariableFloat("z"))


    return pyflamegpu.ALIVE

In [23]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_1() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 1

In [24]:
# z字形下楼的算法
# 假设每层楼高3，行人和stairwell的x、y初始一致
# 通过y,z索引，z字形移动，每次移动一小步，遇到转折点y反向

@pyflamegpu.agent_function
def down_stairwell(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    """
    行人在stairwell点z字形下降，每层楼高3
    通过y,z索引，z字形移动
    """
    # 获取当前位置
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")
    # 获取目标stairwell的x,y
    stairwell_x = pyflamegpu.getVariableFloat("target_stairwell_x")
    stairwell_y = pyflamegpu.getVariableFloat("target_stairwell_y")
    # 获取当前z字形方向（1为y正向，-1为y负向）
    if not pyflamegpu.getVariableInt("zigzag_dir"):
        pyflamegpu.setVariableInt("zigzag_dir", 1)

    dire = pyflamegpu.getVariableInt("zigzag_dir")
    # 每步y方向移动距离
    step_y = 1
    # 每步z方向下降距离
    step_z = 0.5
    # z字形的y范围（比如以stairwell_y为中心，上下各1.5）
    y_min = stairwell_y - 3
    y_max = stairwell_y + 3
    # 计算下一步y
    next_y = y + dire * step_y
    # 判断是否到达边界，若到达则反向
    if next_y > y_max:
        next_y = y_max
        dire = -1
    elif next_y < y_min:
        next_y = y_min
        dire = 1
    # 计算下一步z
    next_z = z - step_z
    # 判断是否到达下一层（z是否小于目标z）
    # 假设目标z为0
    if next_z < 0:
        next_z = 0
        pyflamegpu.setVariableInt("evacuate_status", 3)
    # 更新变量
    pyflamegpu.setVariableFloat("y", next_y)
    pyflamegpu.setVariableFloat("z", next_z)
    pyflamegpu.setVariableInt("zigzag_dir", dire)
    # x保持不变
    pyflamegpu.setVariableFloat("x", stairwell_x)
    # 输出当前位置
    message_out.setLocation(stairwell_x, next_y, next_z)
    return pyflamegpu.ALIVE

    

In [25]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_2() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 2

In [26]:
@pyflamegpu.agent_function
def move_to_stop(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageNone):
    outstop_x = pyflamegpu.getVariableFloat("outstop_x")
    outstop_y = pyflamegpu.getVariableFloat("outstop_y")
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    
    delta_x = outstop_x - x
    delta_y = outstop_y - y

    distance = math.sqrtf(delta_x * delta_x + delta_y * delta_y)

    next_x=0.0
    next_y=0.0
    if distance > 5.0:
        step_x = delta_x / distance 
        step_y = delta_y / distance
        next_x = x + 5*step_x
        next_y = y + 5*step_y
    else:
        next_x =  outstop_x
        next_y =  outstop_y
        pyflamegpu.setVariableInt("evacuate_status", 4)

    pyflamegpu.setVariableFloat("x", next_x)
    pyflamegpu.setVariableFloat("y", next_y)

    return pyflamegpu.ALIVE

In [27]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_3() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 3

In [28]:
@pyflamegpu.agent_function
def move_to_shelter(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    
    fgraph = pyflamegpu.environment.getDirectedGraph("fgraph")
    i = pyflamegpu.getVariableInt("path_point")
    m = pyflamegpu.getVariableIntArray20("shortest_path", i)
    n = pyflamegpu.getVariableIntArray20("shortest_path", i+1)

    target_shelter_x = fgraph.getVertexPropertyFloatArray2("bar", m-1, 0)
    target_shelter_y = fgraph.getVertexPropertyFloatArray2("bar", m-1, 1)
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")


    delta_x = target_shelter_x - x
    delta_y = target_shelter_y - y

    distance = math.sqrtf(delta_x * delta_x + delta_y * delta_y)

    next_x=0.0
    next_y=0.0
    if distance > 5.0:
        step_x = delta_x / distance 
        step_y = delta_y / distance
        next_x = x + 5*step_x
        next_y = y + 5*step_y
    else:
        next_x =  target_shelter_x
        next_y =  target_shelter_y
        if n == -1:
            pyflamegpu.setVariableInt("evacuate_status", 5)
        else:
            pyflamegpu.setVariableInt("path_point", i+1)

    pyflamegpu.setVariableFloat("x", next_x)
    pyflamegpu.setVariableFloat("y", next_y)

    message_out.setLocation(next_x, next_y, pyflamegpu.getVariableFloat("z"))




    return pyflamegpu.ALIVE

In [29]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_4() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 4

@pyflamegpu.agent_function_condition
def eva_state_is_5() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 5

@pyflamegpu.agent_function_condition
def should_output_arrival() -> bool :
    return (pyflamegpu.getVariableInt("accepted") == 1) and (pyflamegpu.getVariableInt("arrival_sent") == 0)

In [30]:

output_func_translated = pyflamegpu.codegen.translate(output_message)
stairwell_output_func_translated = pyflamegpu.codegen.translate(stairwell_output_message)
shelter_output_func_translated = pyflamegpu.codegen.translate(shelter_output_message)
set_target_stairwell_func_translated = pyflamegpu.codegen.translate(set_target_stairwell)
set_target_shelter_func_translated = pyflamegpu.codegen.translate(set_target_shelter_first)

# tutor: set nearest stairwell -> choose nearest shelter by network shortest path
tutor_set_target_stairwell_func_translated = pyflamegpu.codegen.translate(tutor_set_target_stairwell)
tutor_set_target_shelter_by_network_func_translated = pyflamegpu.codegen.translate(tutor_set_target_shelter_by_network)

# tutor 引导消息：tutor 广播 / student 接收
tutor_broadcast_shelter_translated = pyflamegpu.codegen.translate(tutor_broadcast_shelter)
student_follow_tutor_translated = pyflamegpu.codegen.translate(student_follow_tutor)

#ExampleFn_translated = pyflamegpu.codegen.translate(ExampleFn)
move_to_stairwell_func_translated = pyflamegpu.codegen.translate(move_to_stairwell)


ShortestPathFn_translated = pyflamegpu.codegen.translate(ShortestPathFn)
down_stairwell_func_translated = pyflamegpu.codegen.translate(down_stairwell)
move_to_stop_func_translated = pyflamegpu.codegen.translate(move_to_stop)

eva_state_is_1_func_translated = pyflamegpu.codegen.translate(eva_state_is_1)
eva_state_is_2_func_translated = pyflamegpu.codegen.translate(eva_state_is_2)
eva_state_is_3_func_translated = pyflamegpu.codegen.translate(eva_state_is_3)
eva_state_is_4_func_translated = pyflamegpu.codegen.translate(eva_state_is_4)
eva_state_is_5_func_translated = pyflamegpu.codegen.translate(eva_state_is_5)
should_output_arrival_translated = pyflamegpu.codegen.translate(should_output_arrival)
is_set_shortest_path_is_1_func_translated = pyflamegpu.codegen.translate(is_set_shortest_path_is_1)

move_to_shelter_translated = pyflamegpu.codegen.translate(move_to_shelter)

# shelter 容量机制
shelter_update_occupancy_translated = pyflamegpu.codegen.translate(shelter_update_occupancy)
check_shelter_capacity_and_replan_translated = pyflamegpu.codegen.translate(check_shelter_capacity_and_replan)
output_arrival_message_translated = pyflamegpu.codegen.translate(output_arrival_message)


# Setup the two agent functions
output_fn = student_agent.newRTCFunction("output_message", output_func_translated)
output_fn.setMessageOutput("location")

stairwell_output_fn = stairwell_agent.newRTCFunction("stairwell_output_message", stairwell_output_func_translated)
stairwell_output_fn.setMessageOutput("stairwell_location")

shelter_output_fn = shelter_agent.newRTCFunction("shelter_output_message", shelter_output_func_translated)
shelter_output_fn.setMessageOutput("shelter_location")

# shelter：统计到达人数 -> 更新 occupancy/available
shelter_update_fn = shelter_agent.newRTCFunction("shelter_update_occupancy", shelter_update_occupancy_translated)
shelter_update_fn.setMessageInput("arrival")

# student：到达 shelter 后验满员/改选；被接纳后发 arrival 消息
student_check_shelter_fn = student_agent.newRTCFunction("check_shelter_capacity_and_replan", check_shelter_capacity_and_replan_translated)
student_check_shelter_fn.setMessageInput("shelter_location")
student_check_shelter_fn.setRTCFunctionCondition(eva_state_is_5_func_translated)

student_arrival_output_fn = student_agent.newRTCFunction("output_arrival_message", output_arrival_message_translated)
student_arrival_output_fn.setMessageOutput("arrival")
student_arrival_output_fn.setRTCFunctionCondition(should_output_arrival_translated)

# tutor：到达 shelter 后验满员/改选；被接纳后发 arrival 消息
tutor_check_shelter_fn = tutor_agent.newRTCFunction("check_shelter_capacity_and_replan", check_shelter_capacity_and_replan_translated)
tutor_check_shelter_fn.setMessageInput("shelter_location")
tutor_check_shelter_fn.setRTCFunctionCondition(eva_state_is_5_func_translated)

tutor_arrival_output_fn = tutor_agent.newRTCFunction("output_arrival_message", output_arrival_message_translated)
tutor_arrival_output_fn.setMessageOutput("arrival")
tutor_arrival_output_fn.setRTCFunctionCondition(should_output_arrival_translated)

set_target_stairwell_fn = student_agent.newRTCFunction("set_target_stairwell", set_target_stairwell_func_translated)
set_target_stairwell_fn.setMessageInput("stairwell_location")

set_target_shelter_fn = student_agent.newRTCFunction("set_target_shelter", set_target_shelter_func_translated)
set_target_shelter_fn.setMessageInput("shelter_location")

# tutor: message inputs 与 student 相同，但内部决策不同
tutor_set_target_stairwell_fn = tutor_agent.newRTCFunction("tutor_set_target_stairwell", tutor_set_target_stairwell_func_translated)
tutor_set_target_stairwell_fn.setMessageInput("stairwell_location")

tutor_set_target_shelter_fn = tutor_agent.newRTCFunction("tutor_set_target_shelter_by_network", tutor_set_target_shelter_by_network_func_translated)
tutor_set_target_shelter_fn.setMessageInput("shelter_location")

# tutor 广播引导信息（3D 小范围消息）
tutor_broadcast_fn = tutor_agent.newRTCFunction("tutor_broadcast_shelter", tutor_broadcast_shelter_translated)
tutor_broadcast_fn.setMessageOutput("tutor_guidance")

# student 接收 tutor 引导：到达 outstop 前允许改选，并重置最短路（约束在函数内部判断 evacuate_status）
student_follow_tutor_fn = student_agent.newRTCFunction("student_follow_tutor", student_follow_tutor_translated)
student_follow_tutor_fn.setMessageInput("tutor_guidance")

# tutor: 移动方式与 student 完全一致（复用同一套函数/条件）
tutor_ShortestPathFn_fn = tutor_agent.newRTCFunction("ShortestPathFn", ShortestPathFn_translated)
tutor_ShortestPathFn_fn.setRTCFunctionCondition(is_set_shortest_path_is_1_func_translated)

tutor_move_to_stairwell_fn = tutor_agent.newRTCFunction("move_to_stairwell", move_to_stairwell_func_translated)
tutor_move_to_stairwell_fn.setMessageOutput("location")
tutor_move_to_stairwell_fn.setRTCFunctionCondition(eva_state_is_1_func_translated)

tutor_down_stairwell_fn = tutor_agent.newRTCFunction("down_stairwell", down_stairwell_func_translated)
tutor_down_stairwell_fn.setMessageOutput("location")
tutor_down_stairwell_fn.setRTCFunctionCondition(eva_state_is_2_func_translated)

tutor_move_to_stop_fn = tutor_agent.newRTCFunction("move_to_stop", move_to_stop_func_translated)
tutor_move_to_stop_fn.setRTCFunctionCondition(eva_state_is_3_func_translated)

tutor_move_to_shelter_fn = tutor_agent.newRTCFunction("move_to_shelter", move_to_shelter_translated)
tutor_move_to_shelter_fn.setMessageOutput("location")
tutor_move_to_shelter_fn.setRTCFunctionCondition(eva_state_is_4_func_translated)

#ExampleFn_fn = agent.newRTCFunction("ExampleFn",ExampleFn_translated)
ShortestPathFn_fn = student_agent.newRTCFunction("ShortestPathFn",ShortestPathFn_translated)
ShortestPathFn_fn.setRTCFunctionCondition(is_set_shortest_path_is_1_func_translated)

move_to_stairwell_fn = student_agent.newRTCFunction("move_to_stairwell", move_to_stairwell_func_translated)
move_to_stairwell_fn.setMessageOutput("location")
move_to_stairwell_fn.setRTCFunctionCondition(eva_state_is_1_func_translated)

down_stairwell_fn = student_agent.newRTCFunction("down_stairwell", down_stairwell_func_translated)
down_stairwell_fn.setMessageOutput("location")
down_stairwell_fn.setRTCFunctionCondition(eva_state_is_2_func_translated)

move_to_stop_fn = student_agent.newRTCFunction("move_to_stop", move_to_stop_func_translated)
move_to_stop_fn.setRTCFunctionCondition(eva_state_is_3_func_translated)



move_to_shelter_fn = student_agent.newRTCFunction("move_to_shelter", move_to_shelter_translated)
move_to_shelter_fn.setMessageOutput("location")
move_to_shelter_fn.setRTCFunctionCondition(eva_state_is_4_func_translated)


model.addInitFunction(directed_graph_hostfn())
#model.addExecutionRoot(ExampleFn_fn)
model.addExecutionRoot(output_fn)
stairwell_output_fn.dependsOn(output_fn)
shelter_output_fn.dependsOn(output_fn)

# student set
set_target_shelter_fn.dependsOn(shelter_output_fn)
set_target_stairwell_fn.dependsOn(stairwell_output_fn)

# tutor set（先楼梯、再路网选 shelter）
tutor_set_target_stairwell_fn.dependsOn(stairwell_output_fn)
tutor_set_target_shelter_fn.dependsOn(tutor_set_target_stairwell_fn)
tutor_set_target_shelter_fn.dependsOn(shelter_output_fn)

# tutor 广播（到达 outstop 前持续发）
tutor_broadcast_fn.dependsOn(tutor_set_target_shelter_fn)

# student 接收 tutor 引导（每步都能收；函数内部保证 outstop 后不改）
student_follow_tutor_fn.dependsOn(tutor_broadcast_fn)
student_follow_tutor_fn.dependsOn(set_target_shelter_fn)
student_follow_tutor_fn.dependsOn(set_target_stairwell_fn)

# tutor path & move（与 student 相同）
tutor_ShortestPathFn_fn.dependsOn(tutor_set_target_shelter_fn)
tutor_ShortestPathFn_fn.dependsOn(tutor_set_target_stairwell_fn)
tutor_move_to_stairwell_fn.dependsOn(tutor_ShortestPathFn_fn)

tutor_down_stairwell_fn.dependsOn(tutor_move_to_stairwell_fn)
tutor_move_to_stop_fn.dependsOn(tutor_down_stairwell_fn)
tutor_move_to_shelter_fn.dependsOn(tutor_move_to_stop_fn)

# student path & move
# 先允许接收 tutor 引导再进行最短路规划（若改选，会把 is_set_shortest_path 置 0）
ShortestPathFn_fn.dependsOn(student_follow_tutor_fn)
ShortestPathFn_fn.dependsOn(set_target_stairwell_fn)
move_to_stairwell_fn.dependsOn(ShortestPathFn_fn)

down_stairwell_fn.dependsOn(move_to_stairwell_fn)
move_to_stop_fn.dependsOn(down_stairwell_fn)
move_to_shelter_fn.dependsOn(move_to_stop_fn)

# =========================
# 容量机制：到达后验满员/改选 + shelter 计数
# =========================

# student：到达后先验满员 -> 若接纳则发 arrival
student_check_shelter_fn.dependsOn(move_to_shelter_fn)
student_check_shelter_fn.dependsOn(shelter_output_fn)
student_arrival_output_fn.dependsOn(student_check_shelter_fn)

# tutor：到达后先验满员 -> 若接纳则发 arrival
tutor_check_shelter_fn.dependsOn(tutor_move_to_shelter_fn)
tutor_check_shelter_fn.dependsOn(shelter_output_fn)
tutor_arrival_output_fn.dependsOn(tutor_check_shelter_fn)

# shelter：最后统计本步到达（下一步再广播最新 available）
shelter_update_fn.dependsOn(student_arrival_output_fn)
shelter_update_fn.dependsOn(tutor_arrival_output_fn)
shelter_update_fn.dependsOn(shelter_output_fn)


model.generateLayers() 




# Specify the desired StepLoggingConfig
step_log_cfg = pyflamegpu.StepLoggingConfig(model)
# Log every step
step_log_cfg.setFrequency(1)
# Include the mean of the "point" agent population's variable 'drift'
step_log_cfg.agent("student_agent").logMeanInt("accepted")
step_log_cfg.agent("student_agent", "default").logCount()
step_log_cfg.agent("tutor_agent", "default").logCount()

# Create and init the simulation
cuda_model = pyflamegpu.CUDASimulation(model)



## 我之后要干什么？


1. 增加引导员agent，疏散模式。确定引导员agent的各种移动方式。收取
2. 增加stairwell的数量统计。和拥挤统计。
3. 增加shelter死亡的消息。如果shelter死了，找其他的shelter前往。
4. 还要获取常规方法下的最近距离的分配

## GA

In [31]:
import numpy as np

In [32]:
def EvaluateShleter(binary_chromosome):
    """ 
    评估函数：二进制染色体(0/1) -> 避难所索引列表
    约束：binary_chromosome 长度=70，且1的数量=6
    """
    from flamegpu_init_code import initialize_student_agent_population
    from shelter_flamegpu_init_code import initialize_shelter_agent_population
    from tutor_init_code import initialize_tutor_agent_population

    # 从二进制染色体中提取选中的避难所索引
    shelter_list = [i for i, bit in enumerate(binary_chromosome) if bit == 1]

    # initialize agent population
    initialize_student_agent_population(model, cuda_model)
    initialize_tutor_agent_population(model, cuda_model)
    initialize_shelter_agent_population(model, cuda_model, shelter_list) 

    cuda_model.setStepLog(step_log_cfg)
    cuda_model.SimulationConfig().steps = 200 
    cuda_model.CUDAConfig().device_id = 0
    cuda_model.applyConfig()

    cuda_model.simulate()
    run_log = cuda_model.getRunLog()
    step_log = run_log.getStepLog()

    # 获取所有step的 "student_agent" 的 "in_shelter" 平均值组成的列表
    agent_speed_mean_list = [log.getAgent("student_agent").getMean("accepted") for log in step_log]

    # 找到大于0.95的最小索引（Python索引从0开始，如果想要第几个step，加1）
    found_index = next((i for i, x in enumerate(agent_speed_mean_list) if x > 0.95), None)
    if found_index is not None:
        min_index = found_index + 1
    else:
        min_index = 200

    return [int(min_index)]


In [33]:
# 不再需要 array：改用二进制编码(list[int]) 作为染色体


In [34]:
# -------- 二进制(0/1)染色体：长度71，且恰好6个1 --------
# 注意：重复运行该单元格时，deap.creator 可能因重复 create 抛错；这里做了幂等保护。

if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

num_total_shelters = 70
num_select_shelter = 6

def create_binary_individual():
    """创建一个二进制个体：长度70，且恰好6个1"""
    ind = [0] * num_total_shelters
    for idx in random.sample(range(num_total_shelters), num_select_shelter):
        ind[idx] = 1
    return ind

toolbox.register("individual", tools.initIterate, creator.Individual, create_binary_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)


In [35]:
def _repair_to_k_ones(individual, k):
    """修复：强制 individual 中1的数量等于k（就地修改）"""
    ones = [i for i, b in enumerate(individual) if b == 1]
    zeros = [i for i, b in enumerate(individual) if b == 0]
    if len(ones) > k:
        for i in random.sample(ones, len(ones) - k):
            individual[i] = 0
    elif len(ones) < k:
        for i in random.sample(zeros, k - len(ones)):
            individual[i] = 1
    return individual


def binary_crossover(ind1, ind2):
    """单点交叉 + 修复，保证每条染色体仍然恰好6个1"""
    size = len(ind1)
    cxpoint = random.randint(1, size - 1)
    ind1[cxpoint:], ind2[cxpoint:] = ind2[cxpoint:], ind1[cxpoint:]
    _repair_to_k_ones(ind1, num_select_shelter)
    _repair_to_k_ones(ind2, num_select_shelter)
    return ind1, ind2


def binary_mutate(individual, indpb=0.05):
    """通过(1位<->0位)交换的方式变异，天然保持1的数量不变"""
    # 按位概率触发多次交换
    for _ in range(len(individual)):
        if random.random() < indpb:
            ones = [i for i, b in enumerate(individual) if b == 1]
            zeros = [i for i, b in enumerate(individual) if b == 0]
            if not ones or not zeros:
                continue
            i1 = random.choice(ones)
            i0 = random.choice(zeros)
            individual[i1] = 0
            individual[i0] = 1
    return (individual,)


toolbox.register("mate", binary_crossover)
toolbox.register("mutate", binary_mutate, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", EvaluateShleter)


In [36]:
def opt_shelter(population, toolbox, mate_probability, mutate_probability, ngen, stats=None, halloffame=None, log_txt_path=None):
    def binary_to_indices(binary_chrom):
        return [i for i, bit in enumerate(binary_chrom) if bit == 1]

    def _append_txt(line: str):
        if not log_txt_path:
            return
        with open(log_txt_path, "a", encoding="utf-8") as f:
            f.write(line + "\n")
            f.flush()

    logbook = tools.Logbook()
    logbook.header = ['gen', 'nevals'] + (stats.fields if stats else [])

    # 记录每一代的前三最优个体 [(gen, [idx_list...], (fitness,...)), ...]
    top_individuals_all_gens = []

    # 记录到目前为止的全局前三个个体，保存为三元组 (chromosome(binary list), fitness, generation_found)
    overall_top3 = []

    invalid_individuals = [ind for ind in population if not ind.fitness.valid]
    fitnesses = toolbox.map(toolbox.evaluate, invalid_individuals)
    for ind, fit in zip(invalid_individuals, fitnesses):
        ind.fitness.values = fit

    if halloffame is not None:
        halloffame.update(population)

    # 初始化第一代前三最优个体
    current_top3 = tools.selBest(population, 3)
    top_individuals_all_gens.append(
        (0, [binary_to_indices(ind) for ind in current_top3], [ind.fitness.values for ind in current_top3])
    )

    # 更新整体 top3
    for ind in current_top3:
        chrom = list(ind[:])
        fit = ind.fitness.values
        overall_top3.append((chrom, fit, 0))
    overall_top3.sort(key=lambda x: x[1])
    overall_top3 = overall_top3[:3]

    record = stats.compile(population) if stats else {}
    logbook.record(generation=0, nevals=len(invalid_individuals), **record)

    print(logbook.stream)
    print("第0代 当前最优的三个染色体：")
    for idx, ind in enumerate(current_top3):
        print(f"   Top{idx+1}: 染色体索引: {binary_to_indices(ind)} 适应度: {ind.fitness.values}")

    print("至目前为止整体最优的三个染色体：")
    for idx, (chrom, fit, gen_found) in enumerate(overall_top3):
        print(f"   Overall Top{idx+1}: 染色体索引: {binary_to_indices(chrom)} 适应度: {fit} 发现于第{gen_found}代")

    # 写入第 0 代（txt）
    _append_txt(
        f"gen=0 nevals={len(invalid_individuals)} stats={record} "
        f"top3={[ (binary_to_indices(ind), ind.fitness.values) for ind in current_top3 ]} "
        f"overall_top3={overall_top3}"
    )

    for gen in range(1, ngen + 1):
        offspring = toolbox.select(population, len(population))
        offspring = [toolbox.clone(ind) for ind in offspring]

        # 交叉
        for i in range(1, len(offspring), 2):
            if random.random() < mate_probability:
                offspring[i - 1], offspring[i] = toolbox.mate(offspring[i - 1], offspring[i])
                del offspring[i - 1].fitness.values, offspring[i].fitness.values

        # 变异
        for i in range(len(offspring)):
            if random.random() < mutate_probability:
                offspring[i], = toolbox.mutate(offspring[i])
                del offspring[i].fitness.values

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        if halloffame is not None:
            halloffame.update(offspring)

        # 当前代的前三最优个体
        current_top3 = tools.selBest(offspring, 3)
        top_individuals_all_gens.append(
            (gen, [binary_to_indices(ind) for ind in current_top3], [ind.fitness.values for ind in current_top3])
        )

        # 更新整体 top3
        for ind in current_top3:
            chrom = list(ind[:])
            fit = ind.fitness.values
            if not any(chrom == ex_chrom for (ex_chrom, _, _) in overall_top3):
                overall_top3.append((chrom, fit, gen))
        overall_top3.sort(key=lambda x: x[1])
        overall_top3 = overall_top3[:3]

        print(f"第{gen}代 当前最优的三个染色体：")
        for idx, ind in enumerate(current_top3):
            print(f"   Top{idx+1}: 染色体索引: {binary_to_indices(ind)} 适应度: {ind.fitness.values}")

        print("至目前为止整体最优的三个染色体：")
        for idx, (chrom, fit, gen_found) in enumerate(overall_top3):
            print(f"   Overall Top{idx+1}: 染色体索引: {binary_to_indices(chrom)} 适应度: {fit} 发现于第{gen_found}代")

        population[:] = offspring

        record = stats.compile(population) if stats else {}
        logbook.record(gen=gen, nevals=len(invalid_ind), **record)
        print(logbook.stream)


        _append_txt(
            f"gen={gen} nevals={len(invalid_ind)} stats={record} "
            f"top3={[ (binary_to_indices(ind), ind.fitness.values) for ind in current_top3 ]} "
            f"overall_top3={overall_top3}"
        )

    return {
        "per_gen_top3": top_individuals_all_gens,
        "overall_top3": overall_top3,
        "logbook": list(logbook),
        "log_txt_path": log_txt_path,
    }


In [37]:
#实例化一个hof，储存一个最优解
#所以我们还要设计一个可视化点的函数
hof = tools.HallOfFame(1)
population = toolbox.population(n=60)

#很方便：输入的是避难所的索引，得到的是适应度的值的统计量
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)



In [38]:
import datetime
import uuid
from pathlib import Path

# 每次运行创建一个独立实验目录
run_dir = Path("experiments") / "ga_shelter" / (datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "_" + uuid.uuid4().hex[:8])
run_dir.mkdir(parents=True, exist_ok=True)
log_txt_path = run_dir / "gen_log.txt"

# 记录一行 header（可选）
with open(log_txt_path, "a", encoding="utf-8") as f:
    f.write(f"0/1 encoding")
    f.write(f"# GA run_dir={run_dir}\n")
    f.write(f"# mate=0.4 mutate=0.2 ngen=200 pop_n={len(population)} tournsize=3\n")

results = opt_shelter(population, toolbox, 0.4, 0.2, 200, stats=stats, halloffame=hof, log_txt_path=str(log_txt_path))
print("✅ 每代日志已保存到:", log_txt_path)
print("✅ 最终 overall_top3:", results.get("overall_top3"))


初始化 6391 个学生代理个体
学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成
初始化 107 个 tutor 代理个体
tutor 代理种群初始化完成
初始化 6 个shelter代理个体
shelter代理种群初始化完成
初始化 6391 个学生代理个体
学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成
初始化 107 个 tutor 代理个体
tutor 代理种群初始化完成
初始化 6 个shelter代理个体
shelter代理种群初始化完成
初始化 6391 个学生代理个体
学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成
初始化 107 个 tutor 代理个体
tutor 代理种群初始化完成
初始化 6 个shelter代理个体
shelter代理种群初始化完成
初始化 6391 个学生代理个体
学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成
初始化 107 个 tutor 代理个体
tutor 代理种群初始化完成
初始化 6 个shelter代理个体
shelter代理种群初始化完成
初始化 6391 个学生代理个体
学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成
初始化 107 个 tutor 代理个体
tutor 代理种群初始化完成
初始化 6 个shelter代理个体
shelter代理种群初始化完成
初始化 6391 个学生代理个体
学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成
初始化 107 个 tutor 代理个体
tutor 代理种群初始化完成
初始化 6 个shelter代理个体
shelter代理种群初始化完成
初始化 6391 个学生代理个体
学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成
初始化 107 个 tutor 代理个体
tutor 代理种群初始化完成
初始化 6 个shelter代理个体
shelter代理种群初始化完成
初始化 6391 个学生代理个体
学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成
初始化 107 个 tutor 代理个体
tutor 代理种群初始化完成
初始化 